<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Regression discontinuity

**[Regression discontinuity](https://learning.nextia-ai.com/courses/experiments/m05/regression-discontinuity/)** · Experiments and Causal Impact: Did Our Change Work? · Module 5, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate the effect of Larkfield's Gold tier at its 500-dollar threshold: fit a line on each side, read the jump, find the customers who gamed the threshold, and leave them out with a donut.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | a straight line, from the [mathematics course](https://learning.nextia-ai.com/courses/math/); difference-in-differences, from [Difference-in-differences](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/regression-discontinuity/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M05-L03-regression-discontinuity/regression-discontinuity-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Simulate the loyalty tier

This notebook makes its own data with the same code as the project's simulator (`sim run --scenario loyalty`, seed 40). All of it is synthetic. Customers who spent 500 dollars or more in 2024 became Gold on 1 January 2025. The simulator knows both worlds for every customer: with Gold and without it.

The scenario has a practice problem we made on purpose. Do not read the code for it yet: find it in the data.

In [ ]:
def loyalty(seed=40, n=50_000, threshold=500.0, tier_effect=45.0):
    """A copy of the project's scenarios/loyalty.py."""
    rng = np.random.default_rng(seed)
    spend = np.round(rng.lognormal(np.log(330), 0.65, n), 2)
    tenure = rng.integers(200, 2500, n)
    near = (spend >= 455) & (spend < threshold)
    gamer = near & (rng.random(n) < 0.45)
    spend = np.where(gamer, np.round(threshold + rng.uniform(1, 35, n), 2), spend)
    base = 18 + 0.42 * spend - 0.00011 * spend**2 + 0.004 * tenure
    y0 = np.maximum(base * rng.lognormal(-0.02, 0.2, n) - 60 * gamer, 0)
    gold = spend >= threshold
    y = np.where(gold, y0 + tier_effect + 0.03 * (spend - threshold), y0)
    return pd.DataFrame({"spend_2024": spend, "tenure_days": tenure, "gold": gold.astype(int),
                         "spend_2025_h1": np.round(y, 2)})

df = loyalty()
naive = df[df.gold == 1].spend_2025_h1.mean() - df[df.gold == 0].spend_2025_h1.mean()
print(f"{len(df):,} customers, {df.gold.mean():.1%} Gold")
print(f"naive {naive:+.2f} dollars; true effect at 500 dollars: +45.00")

> **Predict.** Before you run the cell: how far is the naive answer from the truth?

> **Check.** You should see `naive +191.73 dollars`.

Gold customers spent more in 2024: that is how they became Gold. So they spend more in 2025, with or without the tier.

## 2. Six customers, by hand

Six made-up customers near the line. On each side, 2025 spending rises by 4 dollars for every 10 dollars of 2024 spending. Extend each side to 500.

In [ ]:
below_at_500 = 190 + 0.4 * (500 - 490)   # from the customer at 490
above_at_500 = 240 - 0.4 * (510 - 500)   # from the customer at 510
print(f"jump {above_at_500 - below_at_500:+.0f}")

> **Check.** You should see `jump +42`.

## 3. A line on each side, bandwidth 100

The regression `spend_2025_h1 ~ above * xc` fits one line below 500 and one above. `xc` is the distance to the threshold, so the coefficient of `above` is the jump at 500. The errors are robust to unequal spread (`HC1`).

In [ ]:
near = df[(df.spend_2024 >= 400) & (df.spend_2024 < 600)].copy()
near["xc"] = near.spend_2024 - 500
near["above"] = (near.spend_2024 >= 500).astype(int)
m = smf.ols("spend_2025_h1 ~ above * xc", data=near).fit(cov_type="HC1")
b, se = m.params["above"], m.bse["above"]
print(f"jump {b:+.2f} (95% interval {b - 1.96 * se:+.2f} to {b + 1.96 * se:+.2f}), {len(near):,} customers")

> **Check.** You should see `jump +17.54 (95% interval +13.84 to +21.25), 10,125 customers`.

The interval does not contain the true +45. Something is wrong near the line.

## 4. Count the customers on each side

Without gaming, the number of customers changes smoothly at 500. Count the customers in the 20 dollars on each side, and draw them in 10-dollar bands.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import binomtest

below = int(((df.spend_2024 >= 480) & (df.spend_2024 < 500)).sum())
above = int(((df.spend_2024 >= 500) & (df.spend_2024 < 520)).sum())
print(below, above, f"p = {binomtest(above, below + above, 0.5).pvalue:.3g}")
df.spend_2024[(df.spend_2024 >= 400) & (df.spend_2024 < 600)].plot.hist(bins=range(400, 610, 10), figsize=(9, 3))
plt.axvline(500, color="grey", linestyle="--")
plt.xlabel("2024 spending (dollars)")
plt.show()

> **Check.** You should see `520 1534 p = 1.49e-115`.

Three times as many customers sit just above 500 as just below. Some customers added a small order to reach Gold. In the simulator, they are bargain hunters who spend less later: the practice problem. The pile-up ends at about 540 dollars.

## 5. Your turn: a 40-dollar donut

Leave out the customers from 500 to 540 (a **donut**), and fit the two lines again on the rest of `near`.

> **Your turn.** Put the jump, rounded to 2 decimals, in `jump_donut`.

In [ ]:
d = near[(near.spend_2024 < 500) | (near.spend_2024 >= 540)]
m = smf.ols("spend_2025_h1 ~ above * xc", data=d).fit(cov_type="HC1")
jump_donut = round(float(m.params["above"]), 2)
print(jump_donut)

In [ ]:
expect('jump_donut', jump_donut, 45.66, tolerance=0.0001)

With the donut, the jump is close to the true +45. Over 50 seeds, the course measured a mean of +44.5 with the donut, and an interval that contained the truth in 92% of the runs. Without it: +17.3, and 0%.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/regression-discontinuity/) has the charts, the bandwidth comparison and the knowledge checks that count toward your certificate. In the project: `sim run --scenario loyalty`, then `xp rdd` and `xp rdd --donut 40`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Interrupted time series](https://learning.nextia-ai.com/courses/experiments/m05/interrupted-time-series/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m05/regression-discontinuity/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.